# as-strided-noncontig-source — worked example 2: Downsample every other column with as_strided

> Worked example from [Delta Drills](https://delta-drills.vercel.app). Atom: `as-strided-noncontig-source`.

**This is a worked example — read it, run each cell, and follow the reasoning.** It's study material, so there's nothing to submit here. Delta Drills hands you a hands-on version to complete yourself as you get comfortable with the idea.

## Setup

In [ ]:
import numpy as np
import torch as t
from torch import Tensor

t.manual_seed(0)
np.random.seed(0)

## Concept

_First time on this topic? Run the **Setup** cell above and skim it: every class and helper mentioned below is defined there. You don't need to have done any other drill first._

`as_strided(size, stride)` lets you fabricate any view by hand — you supply the exact shape and the exact storage stride per axis. Picking out every k-th element along an axis just means multiplying that axis's source stride by k. Because no data is copied, the result aliases the parent's storage.

## Worked solution

Goal: from a contiguous `(H, W)` tensor, build a zero-copy view that keeps every *second* column.

1. A contiguous `(H, W)` tensor has stride `(W, 1)`. Read `sH, sW = x.stride()`.
2. To keep every second column, the output has `W // 2` columns (assume W even). Moving one step along the output's column axis must skip **two** source columns, so the output column stride is `2 * sW`.
3. The row axis is unchanged: moving down one output row still skips a full source row, so the output row stride stays `sH`.
4. Call `t.as_strided(x, size=(H, W // 2), stride=(sH, 2 * sW))`. No loop, no copy.
5. We verify against the obvious slice `x[:, ::2]`, which computes the same thing — the strided view should equal it elementwise and share storage (`data_ptr` matches).
6. Printing the first row shows columns 0, 2, 4, ... exactly as expected.

In [ ]:
def strided_downsample(x: Tensor) -> Tensor:
    H, W = x.shape
    sH, sW = x.stride()
    return t.as_strided(x, size=(H, W // 2), stride=(sH, 2 * sW))

x = t.arange(24, dtype=t.float32).reshape(4, 6)
out = strided_downsample(x)
print('out shape   :', tuple(out.shape))
print('first row   :', out[0].tolist())
print('shares store:', out.data_ptr() == x.data_ptr())
print('matches ::2 :', t.equal(out, x[:, ::2]))